In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from datasets import load_dataset, DownloadConfig

download_config = DownloadConfig(
    local_files_only=True,
    cache_dir=".cache",   # optional
)

/localscratch/nnsfn01/anaconda3/envs/north_caucasus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
LANGS = ['af_za', 'am_et', 'ar_eg', 'ast_es', 'az_az', 'be_by', 'bg_bg', 'bn_in', 'ca_es', 'ceb_ph', 'ckb_iq', 'cmn_hans_cn', 'cs_cz', 'cy_gb', 'da_dk', 'de_de', 'el_gr', 'en_us', 'es_419', 'et_ee', 'fa_ir', 'ff_sn', 'fi_fi', 'fr_fr', 'ga_ie', 'gl_es', 'ha_ng', 'he_il', 'hi_in', 'hr_hr', 'hu_hu', 'hy_am', 'id_id', 'it_it', 'ja_jp', 'jv_id', 'ka_ge', 'kk_kz', 'km_kh', 'kn_in', 'ko_kr', 'ky_kg', 'lg_ug', 'lo_la', 'lt_lt', 'lv_lv', 'mi_nz', 'mk_mk', 'ml_in', 'mn_mn', 'mr_in', 'ms_my', 'mt_mt', 'my_mm', 'nb_no', 'ne_np', 'nl_nl', 'ny_mw', 'om_et', 'or_in', 'pa_in', 'pl_pl', 'ps_af', 'pt_br', 'ro_ro', 'ru_ru', 'sl_si', 'sn_zw', 'so_so', 'sv_se', 'sw_ke', 'ta_in', 'te_in', 'tg_tj', 'th_th', 'tr_tr', 'uk_ua', 'ur_pk', 'uz_uz', 'vi_vn', 'wo_sn', 'xh_za', 'yo_ng', 'yue_hant_hk', 'zu_za']

In [4]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

model_path = "models/w2v2-lv-60-espeak-ipa/"
processor = Wav2Vec2Processor.from_pretrained(model_path)
model = Wav2Vec2ForCTC.from_pretrained(model_path)

In [5]:
import os
import csv
import torch
from itertools import islice
from tqdm.auto import tqdm

device = "cuda:0"

model.eval()
model.to(device)


def batch_iterator(iterable, batch_size):
    iterator = iter(iterable)

    while True:
        batch = list(islice(iterator, batch_size))

        if not batch:
            break

        yield batch


output_path = os.path.join("results", os.path.basename(model_path) if model_path[-1] != '/' else os.path.basename(model_path[:-1]))
os.makedirs(output_path, exist_ok=True)

In [6]:
with open(os.path.join(output_path, "predictions.tsv"), "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    # header
    writer.writerow([
        "id",
        "lang",
        "filename",
        "predicted_ipa",
    ])

    for lang in tqdm(LANGS):
        dataset = load_dataset("fleurs", lang, streaming=True, download_config=download_config, trust_remote_code=True)
        test_dataset = dataset["test"]   # streaming dataset
        model.load_adapter(lang)
        for batch in batch_iterator(test_dataset, batch_size=2):
    
            # audio arrays
            audios = [x["audio"]["array"][:360000] for x in batch]
    
            # processor handles padding dynamically
            inputs = processor(
                audios,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True,
            )
    
            inputs = {k: v.to(device) for k, v in inputs.items()}
    
            with torch.no_grad():
                logits = model(**inputs).logits
    
            pred_ids = torch.argmax(logits, dim=-1)
    
            predicted_ipas = processor.batch_decode(pred_ids)
    
            for x, pred_ipa in zip(batch, predicted_ipas):
    
                writer.writerow([
                    x["id"],
                    lang,
                    os.path.basename(x["audio"]["path"]),
                    pred_ipa,
                ])

print(f"Saved predictions to: {output_path}")

100%|█████████████████████████████████████████| 85/85 [1:34:23<00:00, 66.63s/it]

Saved predictions to: results/w2v2-lv-60-espeak-ipa


In [7]:
import os
import glob
import pandas as pd
from jiwer import wer, cer
from tqdm.auto import tqdm

IPA_ROOT = "fleurs_ipa_asr"
PRED_PATH = os.path.join(output_path, "predictions.tsv")

# ---------------------------------------------------
# Load all IPA TSVs
# ---------------------------------------------------

dfs = []

ipa_files = glob.glob(
    os.path.join(IPA_ROOT, "*", "test_sentences_ipa.tsv")
)

for path in tqdm(sorted(ipa_files)):

    lang = os.path.basename(os.path.dirname(path))

    df = pd.read_csv(path, sep="\t")

    df["lang"] = lang

    dfs.append(df)

gt_df = pd.concat(dfs, ignore_index=True)

# rename for merge
gt_df = gt_df.rename(
    columns={
        "audio_file": "filename",
    }
)

# ---------------------------------------------------
# Load predictions
# ---------------------------------------------------

pred_df = pd.read_csv(
    PRED_PATH,
    sep="\t",
)

# ---------------------------------------------------
# Normalize
# ---------------------------------------------------

gt_df["ipa"] = (
    gt_df["ipa"]
    .fillna("")
    .astype(str)
    .str.replace("<unk>", "!", regex=False)
)

pred_df["predicted_ipa"] = (
    pred_df["predicted_ipa"]
    .fillna("")
    .astype(str)
    .str.replace("<unk>", "!", regex=False)
)

# ensure same dtypes
gt_df["id"] = gt_df["id"].astype(int)
pred_df["id"] = pred_df["id"].astype(int)

# ---------------------------------------------------
# Merge
# ---------------------------------------------------

merged = gt_df.merge(
    pred_df,
    how="inner",
    on=["id", "lang", "filename"],
)

print("Merged rows:", len(merged),'<',len(gt_df))

# ---------------------------------------------------
# Compute per-entry metrics
# ---------------------------------------------------

wers = []
cers = []

for ref, hyp in tqdm(
    zip(merged["ipa"], merged["predicted_ipa"]),
    total=len(merged),
):

    wers.append(wer(ref, hyp))
    cers.append(cer(ref, hyp))

merged["wer"] = wers
merged["cer"] = cers

# ---------------------------------------------------
# Aggregate by language
# ---------------------------------------------------

lang_metrics = (
    merged
    .groupby("lang")[["wer", "cer"]]
    .mean()
    .reset_index()
    .sort_values("cer")
)

print(lang_metrics)

# ---------------------------------------------------
# Overall
# ---------------------------------------------------

print("\nOverall:")
print("WER:", merged["wer"].mean())
print("CER:", merged["cer"].mean())

100%|██████████████████████████████████████████| 85/85 [00:00<00:00, 121.81it/s]


Merged rows: 64924 < 64924


100%|███████████████████████████████████| 64924/64924 [00:11<00:00, 5637.45it/s]

           lang       wer       cer
17        en_us  0.234687  0.070025
47        mk_mk  0.629437  0.110788
18       es_419  0.586064  0.134557
70        sw_ke  0.601445  0.135050
22        fi_fi  0.723026  0.139326
..          ...       ...       ...
38        km_kh  0.902701  0.342266
83  yue_hant_hk  0.985235  0.350746
14        da_dk  0.892562  0.353212
24        ga_ie  0.873924  0.392334
82        yo_ng  0.916672  0.400011

[85 rows x 3 columns]

Overall:
WER: 0.7946084462277315
CER: 0.2254724029622105


In [8]:
print(lang_metrics.to_string())

           lang       wer       cer
17        en_us  0.234687  0.070025
47        mk_mk  0.629437  0.110788
18       es_419  0.586064  0.134557
70        sw_ke  0.601445  0.135050
22        fi_fi  0.723026  0.139326
73        tg_tj  0.645459  0.142942
19        et_ee  0.766151  0.144751
33        it_it  0.677204  0.146937
37        kk_kz  0.685443  0.148464
32        id_id  0.654154  0.151026
39        kn_in  0.795679  0.159428
75        tr_tr  0.726048  0.159738
31        hy_am  0.772707  0.159970
41        ky_kg  0.772640  0.160434
9        ceb_ph  0.529896  0.160699
51        ms_my  0.668322  0.160769
29        hr_hr  0.721345  0.163592
25        gl_es  0.688511  0.166922
3        ast_es  0.693956  0.168167
20        fa_ir  0.640154  0.170711
48        ml_in  0.826840  0.175833
5         be_by  0.804607  0.176494
72        te_in  0.796621  0.176937
35        jv_id  0.714251  0.185585
45        lv_lv  0.809926  0.186574
40        ko_kr  0.837785  0.187542
66        sl_si  0.798660  0